# dfs-three-set-toposort — faded example 2: Three-set DFS correctly raises ValueError on a cycle

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `dfs-three-set-toposort`. Running the beacon reports progress on the `Backprop: DFS three-set toposort` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: DFS three-set toposort` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dfs-three-set-toposort`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dfs-three-set-toposort"
DD_SUBTOPIC = "Backprop: DFS three-set toposort"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The `temp` set in the DFS toposort acts as a cycle detector: if we encounter a node that is already on the current DFS stack (present in `temp`), we have found a back edge, which means the graph contains a cycle. A valid topological sort is impossible on a cyclic graph, so the function raises `ValueError`.

## Faded exercise 2

Complete `topological_sort_with_cycle_check`. The graph passed to it contains a cycle (A → B → C → A). Your implementation must raise `ValueError` when the cycle is detected. Fill in the cycle-detection guard inside `visit`.

**Fill in:** The two-line temp-set cycle-detection guard: check if `nid` is already in `temp`, and if so raise `ValueError`.

In [ ]:
def topological_sort_with_cycle_check(root, get_children):
    result = []
    perm = set()
    temp = set()

    def visit(node):
        nid = id(node)
        if nid in perm:
            return
        # --- fill in the cycle guard here ---
        raise NotImplementedError()  # TODO: The two-line temp-set cycle-detection guard: check if `nid` is already in `temp`, and if so raise `ValueError`.
        temp.add(nid)
        for child in get_children(node):
            visit(child)
        temp.discard(nid)
        perm.add(nid)
        result.append(node)

    visit(root)
    return result


def _test():
    class N:
        def __init__(self, name): self.name = name; self.kids = []
        def __repr__(self): return self.name

    A, B, C = N('A'), N('B'), N('C')
    A.kids = [B]
    B.kids = [C]
    C.kids = [A]  # cycle back to A

    import pytest
    try:
        topological_sort_with_cycle_check(A, lambda n: n.kids)
        assert False, 'Expected ValueError for cyclic graph'
    except ValueError:
        pass  # correct

    # Acyclic graph should work fine
    X, Y, Z = N('X'), N('Y'), N('Z')
    X.kids = [Y]
    Y.kids = [Z]
    Z.kids = []
    result = topological_sort_with_cycle_check(X, lambda n: n.kids)
    assert len(result) == 3
    assert result[-1] is X


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def topological_sort_with_cycle_check(root, get_children):
    result = []
    perm = set()
    temp = set()

    def visit(node):
        nid = id(node)
        if nid in perm:
            return
        if nid in temp:
            raise ValueError(f'Cycle detected at {node!r}')
        temp.add(nid)
        for child in get_children(node):
            visit(child)
        temp.discard(nid)
        perm.add(nid)
        result.append(node)

    visit(root)
    return result
```
</details>